# Joint outcomes in the paper's binary `causl` setting

This joint-outcome experiment evaluates causal-margin recovery and outcome dependence.
It includes a shuffled-dependence diagnostic and a conditional-engression
g-computation baseline; it is not a comparison with Frugal Flows.
We extend the paper's 5-instrument/5-confounder binary-treatment setup to two
continuous outcomes. The R package `causl` generates the observed data with a
Gaussian pair copula. The package's joint model uses `noise_dim=2`; when sampling the causal margin,
`StoNet` also supplies a two-dimensional reference `eta`. A separate randomized-treatment diagnostic checks
whether the specified moments agree with the generated model; it is not used
to define the causal truth. Under either intervention `do(A=a)` the specified causal
margin is bivariate normal with exact mean `(2a, -a)`, unit marginal variances,
and cross-covariance `0.4`. The first outcome retains the paper's five
confounder copula links. The R generator and its settings are in
`examples/causl_joint_outcomes.R`. Ground truth is given by the model parameters,
not by reference Monte Carlo samples.

**Run All** generates one `n=5000` dataset, fits a single joint Frengression
model for 1000 updates, then prints both arms' marginal and dependence errors and compares them
with a longer fit. The reporting section adds joint probabilities, a joint-CDF
grid check, an audit of the composed observational model, and an optional fitted
comparison. Replications are required for a comparative paper result.
This requires R with `causl` and the Python environment used for the existing
Frengression notebook (`torch`, `engression`, `numpy`, `pandas`, `matplotlib`, `scipy`).
Install R's package with `remotes::install_github('rje42/causl')` if needed.
The notebook contains no saved results. When run, it saves report tables,
metadata and a figure under `~/frengression_results/joint_outcomes_causl`,
outside this repository. The data generator, architecture and `noise_dim=2`
are unchanged.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import json
import platform
from functools import lru_cache
from importlib import metadata as package_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.integrate import quad
from scipy.stats import norm
from engression.models import StoNet
from engression.loss_func import energy_loss_two_sample

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'frengression').is_dir()
             and (p / 'examples' / 'causl_joint_outcomes.R').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the frengression_pkg repository or examples/ folder.')
sys.path.insert(0, str(ROOT / 'src'))
from frengression import Frengression

RSCRIPT = shutil.which('Rscript')
if RSCRIPT is None:
    raise RuntimeError('Rscript was not found. Install R and the R package causl.')
check = subprocess.run([RSCRIPT, '-e', 'stopifnot(requireNamespace("causl", quietly=TRUE))'],
                       capture_output=True, text=True)
if check.returncode:
    raise RuntimeError('R package causl is missing. In R: remotes::install_github("rje42/causl")'
                       + '\n' + check.stderr[-1000:])
print('Repository:', ROOT)
print('Rscript:', RSCRIPT)
print('torch:', torch.__version__)

## Generate the common observational dataset

The causal truth below is **specified by the model parameters**. The
observational residual check cannot by itself verify the interventional law;
the following randomized diagnostic checks the specification separately. `I1`–`I5` influence treatment, while all
five `X` covariates are associated with treatment and `Y1`.


In [ ]:
COVARIATES = [f'I{i}' for i in range(1, 6)] + [f'X{i}' for i in range(1, 6)]
TRUE_COV = np.array([[1.0, 0.4], [0.4, 1.0]])
TRUE_MEAN = {0: np.array([0.0, 0.0]), 1: np.array([2.0, -1.0])}

def generate_causl_joint(n=5000, seed=2026, strength_instr=1.0, randomized=False):
    with tempfile.TemporaryDirectory(prefix='causl_joint_') as temp:
        csv_path = Path(temp) / 'observed.csv'
        cmd = [RSCRIPT, str(ROOT / 'examples' / 'causl_joint_outcomes.R'),
               str(csv_path), str(n), str(seed), str(strength_instr)]
        if randomized:
            cmd.append('randomized')
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode:
            raise RuntimeError('Native causl generation failed:\n' +
                               run.stderr[-4000:] + '\n' + run.stdout[-1000:])
        frame = pd.read_csv(csv_path)
    expected = COVARIATES + ['A', 'Y1', 'Y2']
    if frame.columns.tolist() != expected or len(frame) != n:
        raise ValueError(f'Unexpected causl columns/rows: {frame.shape}, {frame.columns.tolist()}')
    x = frame[['A']].to_numpy(dtype=np.float32)
    z = frame[COVARIATES].to_numpy(dtype=np.float32)
    y = frame[['Y1', 'Y2']].to_numpy(dtype=np.float32)
    if not np.isfinite(np.concatenate([z, x, y], axis=1)).all() or not np.isin(x, [0, 1]).all():
        raise ValueError('causl produced non-finite values or treatment outside {0,1}')
    if min((x[:, 0] == 0).sum(), (x[:, 0] == 1).sum()) == 0:
        raise ValueError('One observed treatment arm is empty')
    # This sanity check uses the observational residuals, never a simulated truth set.
    residual = y - np.column_stack((2 * x[:, 0], -x[:, 0]))
    empirical_cov = np.cov(residual, rowvar=False)
    if n >= 3000 and abs(empirical_cov[0, 1] - TRUE_COV[0, 1]) > 0.15:
        raise RuntimeError('causl residual covariance does not match the specified 0.4. '
                           f'Observed={empirical_cov[0,1]:.3f}; inspect the R/copula version before fitting.')
    return x, z, y, empirical_cov

N, SEED, STRENGTH_INSTR = 5000, 2026, 1.0
x, z, y, observed_residual_cov = generate_causl_joint(N, SEED, STRENGTH_INSTR)
print('x, z, y shapes:', x.shape, z.shape, y.shape)
print('Observed residual cross-covariance (sanity check):', round(observed_residual_cov[0,1], 3))
print('Specified causal cross-covariance:', TRUE_COV[0,1])


### Check the causal specification independently

The next cell uses the **same `causl` parameters** with randomized treatment
(propensity 1/2) in a separate, larger diagnostic sample. In that sample, each
observed arm approximates the corresponding interventional law. This checks
our interpretation of the two-outcome `causl` copula; its sample moments are
**never used as ground truth** in the fitted-model scores. If they disagree
substantially with the specified mean and covariance, stop and inspect the R
model before fitting.


In [ ]:
x_check, _, y_check, _ = generate_causl_joint(
    n=20000, seed=SEED + 1, strength_instr=STRENGTH_INSTR, randomized=True)
check_rows = []
for arm in (0, 1):
    d = y_check[x_check[:, 0] == arm]
    c = np.cov(d, rowvar=False)
    check_rows.append(dict(arm=arm, count=len(d),
                           mean1=d[:, 0].mean(), expected_mean1=TRUE_MEAN[arm][0],
                           mean2=d[:, 1].mean(), expected_mean2=TRUE_MEAN[arm][1],
                           var1=c[0, 0], var2=c[1, 1],
                           cov12=c[0, 1], expected_cov12=TRUE_COV[0, 1]))
check_table = pd.DataFrame(check_rows)
display(check_table.round(3))
print('Diagnostic only: the exact values in the expected columns define the truth.')


## Fit one joint model

`y_dim=2` and `noise_dim=2` use the package's original joint architecture
(3 layers, width 100, learning rate `1e-4`). `model_y` receives a
2-dimensional reference `eta` and also draws its own 2-dimensional layer
noise, as it does in the scalar experiment. The `noise_dim` parameter is 2.
Crucially, `StoNet.sample` returns `(batch, outcome_dimension, draws)`;
we transpose its axes before calculating any joint moments. The displayed seconds cover only fitting the causal-margin modules.
For a stable paper result, inspect the pilot before setting `RUN_REPEATS=True`.


In [ ]:
FR_ITERS = 1000
MC_DRAWS = 5000
torch.set_num_threads(1)

def fit_joint(x, z, y, seed, iterations=None, draws=None):
    iterations = FR_ITERS if iterations is None else iterations
    draws = MC_DRAWS if draws is None else draws
    torch.manual_seed(seed)
    model = Frengression(x_dim=1, y_dim=2, z_dim=10,
                         num_layer=3, hidden_dim=100, noise_dim=2,
                         x_binary=True, z_binary_dims=0, device=torch.device('cpu'))
    xt, zt, yt = (torch.as_tensor(v, dtype=torch.float32) for v in (x, z, y))
    start = time.perf_counter()
    model.train_y(xt, zt, yt, num_iters=iterations, lr=1e-4,
                  print_every_iter=max(1, iterations // 5), tol=0.0)
    seconds = time.perf_counter() - start
    arm_draws = {}
    for arm in (0, 1):
        sample = model.sample_causal_margin(torch.tensor([arm], dtype=torch.int32),
                                            sample_size=draws)
        # StoNet.sample returns [data_size, response_dim, sample_size].
        # Direct reshape(draws, 2) would scramble the two outcomes.
        if tuple(sample.shape) != (1, y.shape[1], draws):
            raise RuntimeError(f'Unexpected joint sample shape: {tuple(sample.shape)}')
        arm_draws[arm] = sample[0].transpose(0, 1).contiguous().cpu().numpy()
        if not np.isfinite(arm_draws[arm]).all():
            raise ValueError(f'Non-finite Frengression sample for arm {arm}')
    return model, arm_draws, seconds

joint_model, joint_draws, fit_seconds = fit_joint(x, z, y, SEED)
print(f'One joint fit: {fit_seconds:.1f} seconds')
print('Frengression noise_dim=2; joint samples transposed to [draws, outcomes]')

## Compare with the exact specified law

Report marginal means and variances **and** the off-diagonal covariance for
each intervention. The shuffled second coordinate below is a diagnostic:
shuffling keeps both fitted one-dimensional margins exactly the same while
removing their dependence. It is not a fitted competing method and must not
be called a Frugal Flows comparison.


In [ ]:
def evaluate_joint(draws, label):
    rows = []
    for arm in (0, 1):
        d = np.asarray(draws[arm], dtype=float)
        mu, cov = d.mean(axis=0), np.cov(d, rowvar=False)
        rows.append(dict(method=label, arm=arm, true_mean1=TRUE_MEAN[arm][0],
                         estimated_mean1=mu[0], true_mean2=TRUE_MEAN[arm][1],
                         estimated_mean2=mu[1], mean_mae=np.abs(mu - TRUE_MEAN[arm]).mean(),
                         true_var1=1.0, estimated_var1=cov[0,0],
                         true_var2=1.0, estimated_var2=cov[1,1],
                         true_cov12=0.4, estimated_cov12=cov[0,1],
                         cov12_error=abs(cov[0,1] - 0.4),
                         covariance_fro_error=np.linalg.norm(cov - TRUE_COV, ord='fro')))
    return pd.DataFrame(rows)

rng = np.random.default_rng(SEED + 99)
shuffled = {arm: np.column_stack((joint_draws[arm][:, 0],
             rng.permutation(joint_draws[arm][:, 1]))) for arm in (0, 1)}
pilot_table = pd.concat([evaluate_joint(joint_draws, 'frengression_joint'),
                         evaluate_joint(shuffled, 'shuffled_diagnostic')], ignore_index=True)
display(pilot_table.round(4))


### Check sensitivity to training length

Refit the **same data and initialization seed** for 4000 updates to inspect
optimization sensitivity. Keep the original pilot and longer fit visible.
`PAPER_FIT` below selects which training budget the reporting and replication
sections use. Its default is `pilot`, corresponding to the original
`frengression_joint` table. Fix this choice before replications; do not select
a separate best budget for each seed using its true causal scores.

In [ ]:
LONGER_ITERS = 4000
longer_model, longer_draws, longer_seconds = fit_joint(
    x, z, y, SEED, iterations=LONGER_ITERS)
longer_table = evaluate_joint(longer_draws, f'frengression_{LONGER_ITERS}_updates')
comparison = pd.concat([pilot_table.loc[pilot_table.method == 'frengression_joint'],
                        longer_table], ignore_index=True)
display(comparison[['method', 'arm', 'estimated_mean1', 'estimated_mean2',
                    'mean_mae', 'estimated_var1', 'estimated_var2',
                    'estimated_cov12', 'cov12_error']].round(4))
print(f'Longer fit time: {longer_seconds:.1f} seconds')
if (comparison.loc[comparison.method == f'frengression_{LONGER_ITERS}_updates',
                  'mean_mae'].max() > 0.2 or longer_table.cov12_error.max() > 0.2):
    print('The longer fit has substantial moment errors; inspect this training budget before using it.')
else:
    print('The longer fit passes this coarse moment check; inspect calibration and repeat the selected budget.')


# The same selected training budget is used in the report and every replication.
PAPER_FIT = 'pilot'  # 'pilot' or 'longer'; choose before running replications.
if PAPER_FIT == 'pilot':
    paper_model, paper_draws, paper_fit_seconds = joint_model, joint_draws, fit_seconds
    PAPER_ITERS = FR_ITERS
elif PAPER_FIT == 'longer':
    paper_model, paper_draws, paper_fit_seconds = longer_model, longer_draws, longer_seconds
    PAPER_ITERS = LONGER_ITERS
else:
    raise ValueError("PAPER_FIT must be 'pilot' or 'longer'")
print(f'Reporting fit: {PAPER_FIT}, {PAPER_ITERS} updates')

### Inspect sampling and latent calibration

The warning about missing covariate dimensions is emitted when `StoNet`
receives only treatment and fills the reference-eta coordinates itself.
Compare that route with explicitly sampled Gaussian `eta`, check the learned
`h(Z,a,xi)` against a standard bivariate normal for each fixed intervention,
and compare generated observational outcomes with observed outcomes within
each arm. The observational arm moments are **not** the interventional truth.

The additional `causal_via_f_h_empirical_Z` row samples the composed model
`f(a,h(Z,a,xi))` over the full empirical covariate distribution. It audits
whether the observational model implies the same causal margin as direct
sampling of `f(a,eta)`. This audit uses the original empirical covariate law;
it does not establish that a different covariate generator can be substituted
without recalibrating `h`.

In [ ]:
paper_model.eval()
torch.manual_seed(SEED + 501)
rng = np.random.default_rng(SEED + 502)
diagnostic_rows = []
latent_draws = {}
implied_draws = {}
with torch.no_grad():
    xt = torch.as_tensor(x, dtype=torch.float32)
    zt = torch.as_tensor(z, dtype=torch.float32)
    eta_observed = paper_model.model_eta(torch.cat([xt, zt], dim=1))
    y_observed_fit = paper_model.model_y(torch.cat([xt, eta_observed], dim=1)).cpu().numpy()
    for arm in (0, 1):
        mask = x[:, 0] == arm
        x_ref = torch.full((MC_DRAWS, 1), float(arm))
        eta_ref = torch.randn(MC_DRAWS, 2)
        explicit = paper_model.model_y(torch.cat([x_ref, eta_ref], dim=1)).cpu().numpy()
        api = paper_draws[arm]
        # Each z is marginal, independently of the fixed intervention arm.
        z_ref = torch.as_tensor(z[rng.permutation(len(z))], dtype=torch.float32)
        x_latent = torch.full((len(z), 1), float(arm))
        eta_refit = paper_model.model_eta(torch.cat([x_latent, z_ref], dim=1)).cpu().numpy()
        latent_draws[arm] = eta_refit
        implied_draws[arm] = paper_model.model_y(torch.cat([x_latent,
            torch.as_tensor(eta_refit, dtype=torch.float32)], dim=1)).cpu().numpy()
        for label, values in [('causal_via_f_h_empirical_Z', implied_draws[arm]),
                              ('causal_api', api), ('causal_explicit_eta', explicit),
                              ('eta_given_arm', eta_refit),
                              ('observed_data', y[mask]),
                              ('observed_fit', y_observed_fit[mask])]:
            mu = values.mean(axis=0)
            cov = np.cov(values, rowvar=False)
            diagnostic_rows.append(dict(arm=arm, source=label, n=len(values),
                                        mean1=mu[0], mean2=mu[1],
                                        var1=cov[0, 0], var2=cov[1, 1],
                                        cov12=cov[0, 1]))
diagnostic_table = pd.DataFrame(diagnostic_rows)
display(diagnostic_table.round(3))
print('eta_given_arm target: means (0, 0), variances (1, 1), covariance 0.')
print('causal_api and causal_explicit_eta should agree within simulation error.')
print('observed_data versus observed_fit checks model fit, not causal recovery.')

## Joint distribution and joint causal queries

The original moment table checks means, variances and covariance. We now also
check marginal quantiles and the joint CDF on a fixed 5-by-5 grid of thresholds
centered on each arm's specified mean. Grid errors evaluate 25 joint
probabilities; they are not a full-distribution distance or a supremum over
all thresholds. Both the mean and maximum grid error are reported.

Two fixed-threshold events give identifiable joint causal queries:
`Y1>0 and Y2>0`, and `Y1>1 and Y2>-0.5`. We report each arm's probability and
the treatment contrast using the **same thresholds in both arms**. A third
event, both outcomes above their own true arm medians, isolates dependence:
its true probability is `1/4 + asin(0.4)/(2*pi)`, about 0.3155, compared with
0.25 under independent exact margins. Its arm-specific thresholds mean it
is not used as a fixed-event causal contrast.

All probability truths are calculated from the specified Gaussian law using
deterministic numerical integration, not estimated from randomized diagnostic
data. `model_draw_mcse` measures only the Monte Carlo error of draws from a
fixed fitted model. It is not uncertainty due to fitting that model.

In [ ]:
QUANTILE_LEVELS = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
CDF_GRID_OFFSETS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
FIXED_EVENTS = {'both_above_zero': (0.0, 0.0),
                'Y1_above_1_Y2_above_minus_half': (1.0, -0.5)}

def validate_draws(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != 2 or len(values) < 2:
        raise ValueError(f'Expected at least two [draws, 2] joint draws, got {values.shape}')
    if not np.isfinite(values).all():
        raise ValueError('Non-finite joint draws')
    return values

@lru_cache(maxsize=4096)
def standard_normal_joint_cdf(t1, t2, rho):
    if not -1.0 < rho < 1.0:
        raise ValueError('A nonsingular Gaussian correlation is required')
    if abs(rho) < 1e-14:
        return float(norm.cdf(t1) * norm.cdf(t2))
    value, error = quad(lambda u: norm.pdf(u) * norm.cdf(
        (t2 - rho * u) / np.sqrt(1.0 - rho**2)),
        -np.inf, t1, epsabs=1e-10, epsrel=1e-10)
    if error > 1e-7 or not np.isfinite(value):
        raise RuntimeError('Gaussian joint-probability integration failed')
    return float(value)

def gaussian_joint_probability(thresholds, mean, covariance, upper=False):
    mean, covariance = np.asarray(mean), np.asarray(covariance)
    sd = np.sqrt(np.diag(covariance))
    if np.any(sd <= 0) or not np.isfinite(covariance).all():
        raise ValueError('Invalid Gaussian covariance')
    t1, t2 = (np.asarray(thresholds) - mean) / sd
    rho = float(covariance[0, 1] / np.prod(sd))
    # Central symmetry avoids subtracting nearly equal tail probabilities.
    if upper:
        t1, t2 = -t1, -t2
    return standard_normal_joint_cdf(float(t1), float(t2), rho)

def cdf_grid_errors(values, mean, covariance):
    d = validate_draws(values)
    sd = np.sqrt(np.diag(covariance))
    thresholds = np.asarray(mean) + CDF_GRID_OFFSETS[:, None] * sd
    errors = []
    for t1 in thresholds[:, 0]:
        for t2 in thresholds[:, 1]:
            estimate = np.mean((d[:, 0] <= t1) & (d[:, 1] <= t2))
            truth = gaussian_joint_probability((t1, t2), mean, covariance)
            errors.append(abs(estimate - truth))
    return float(np.mean(errors)), float(np.max(errors))

def evaluate_events(draws, label):
    rows = []
    for arm in (0, 1):
        d = validate_draws(draws[arm])
        events = {**FIXED_EVENTS, 'both_above_true_arm_medians': tuple(TRUE_MEAN[arm])}
        for event, thresholds in events.items():
            t1, t2 = thresholds
            estimate = float(np.mean((d[:, 0] > t1) & (d[:, 1] > t2)))
            truth = gaussian_joint_probability(thresholds, TRUE_MEAN[arm], TRUE_COV, upper=True)
            fitted_product = float(np.mean(d[:, 0] > t1) * np.mean(d[:, 1] > t2))
            true_product = gaussian_joint_probability(thresholds, TRUE_MEAN[arm],
                                                      np.diag(np.diag(TRUE_COV)), upper=True)
            rows.append(dict(method=label, arm=arm, event=event, threshold1=t1, threshold2=t2,
                fixed_across_arms=event in FIXED_EVENTS, true_probability=truth,
                estimated_probability=estimate, probability_error=abs(estimate - truth),
                signed_probability_error=estimate - truth,
                true_independent_probability=true_product,
                fitted_independent_probability=fitted_product,
                true_dependence_gap=truth - true_product,
                estimated_dependence_gap=estimate - fitted_product,
                model_draw_mcse=np.sqrt(estimate * (1.0 - estimate) / len(d)), n_draws=len(d)))
    return pd.DataFrame(rows)

def evaluate_joint_full(draws, label):
    moments = evaluate_joint(draws, label)
    rows = []
    events = evaluate_events(draws, label)
    for arm in (0, 1):
        d = validate_draws(draws[arm])
        grid_mae, grid_max = cdf_grid_errors(d, TRUE_MEAN[arm], TRUE_COV)
        true_quantiles = TRUE_MEAN[arm] + norm.ppf(QUANTILE_LEVELS)[:, None] * np.sqrt(np.diag(TRUE_COV))
        rows.append(dict(method=label, arm=arm,
            marginal_quantile_mae=float(np.abs(np.quantile(d, QUANTILE_LEVELS, axis=0) - true_quantiles).mean()),
            joint_cdf_grid_mae=grid_mae, joint_cdf_grid_max_error=grid_max,
            joint_event_mae=float(events.loc[events.arm == arm, 'probability_error'].mean()),
            n_draws=len(d)))
    return moments.merge(pd.DataFrame(rows), on=['method', 'arm'], validate='one_to_one')

def fixed_event_contrasts(events):
    rows = []
    selected = events.loc[events.fixed_across_arms]
    group_keys = ['method', 'event']
    if 'seed' in selected.columns:
        group_keys.insert(0, 'seed')
    for key, group in selected.groupby(group_keys, sort=False):
        arm_rows = group.set_index('arm')
        if set(arm_rows.index) != {0, 1} or len(arm_rows) != 2:
            continue  # A failed or missing arm is not a valid treatment contrast.
        values = key if isinstance(key, tuple) else (key,)
        row = dict(zip(group_keys, values))
        truth = arm_rows.loc[1, 'true_probability'] - arm_rows.loc[0, 'true_probability']
        estimate = arm_rows.loc[1, 'estimated_probability'] - arm_rows.loc[0, 'estimated_probability']
        row.update(true_contrast=truth, estimated_contrast=estimate,
                   contrast_error=abs(estimate - truth), signed_contrast_error=estimate - truth,
                   model_draw_contrast_mcse=float(np.sqrt(np.square(arm_rows.model_draw_mcse).sum())))
        rows.append(row)
    return pd.DataFrame(rows)

def shuffle_second_coordinate(draws, seed):
    rng = np.random.default_rng(seed)
    return {arm: np.column_stack((validate_draws(draws[arm])[:, 0],
                  rng.permutation(validate_draws(draws[arm])[:, 1]))) for arm in (0, 1)}

## A fitted joint comparison: conditional engression plus g-computation

This baseline fits one ordinary joint engression model of `Y | A,Z`, with
the same 3-layer, width-100 network and `noise_dim=2`, learning rate `1e-4`,
and the selected Frengression update count. For each intervention it fixes
`A=a`, draws `Z` from the full empirical covariate distribution, and generates
the two outcomes **together**. Thus it can learn dependence, but does not use
Frengression's explicit causal-margin map and latent calibration objective.

`RUN_GCOMP_PILOT=True` fits this comparison on the same observational dataset.
Equal per-network settings are a transparent starting point, not a claim of
optimal tuning or equal total parameter counts. Report repeated results
before claiming a comparative advantage. Timings cover the fitted
causal-margin modules or conditional outcome model, not complete simulators.

In [ ]:
def fit_conditional_gcomp(x, z, y, seed, iterations, draws=MC_DRAWS):
    torch.manual_seed(seed)
    device = torch.device('cpu')
    model = StoNet(x.shape[1] + z.shape[1], y.shape[1],
                   3, 100, 2, add_bn=False, noise_all_layer=False).to(device)
    inputs = torch.as_tensor(np.column_stack((x, z)), dtype=torch.float32, device=device)
    targets = torch.as_tensor(y, dtype=torch.float32, device=device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.train()
    start = time.perf_counter()
    for iteration in range(iterations):
        optimizer.zero_grad()
        sample1, sample2 = model(inputs), model(inputs)
        loss, _, _ = energy_loss_two_sample(targets, sample1, sample2)
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite conditional-engression loss')
        loss.backward()
        optimizer.step()
        if iteration == 0 or (iteration + 1) % max(1, iterations // 5) == 0:
            print(f'Conditional g-computation update {iteration + 1}: loss {loss.item():.4f}')
    seconds = time.perf_counter() - start
    model.eval()
    rng = np.random.default_rng(seed + 700)
    arm_draws = {}
    with torch.no_grad():
        for arm in (0, 1):
            sampled_z = z[rng.integers(0, len(z), size=draws)]
            reference = np.column_stack((np.full((draws, x.shape[1]), arm), sampled_z))
            outputs = [model(torch.as_tensor(batch, dtype=torch.float32, device=device)).cpu().numpy()
                       for batch in np.array_split(reference, max(1, (draws + 2047) // 2048))]
            arm_draws[arm] = validate_draws(np.concatenate(outputs, axis=0))
    return model, arm_draws, seconds

RUN_GCOMP_PILOT = True
GCOMP_ITERS = PAPER_ITERS
paper_methods = {'frengression_joint': paper_draws,
                 'shuffled_diagnostic': shuffle_second_coordinate(paper_draws, SEED + 99),
                 'frengression_gcomp_audit': implied_draws}
paper_times = {'frengression_joint': paper_fit_seconds,
               'shuffled_diagnostic': np.nan, 'frengression_gcomp_audit': np.nan}
if RUN_GCOMP_PILOT:
    gcomp_model, gcomp_draws, gcomp_seconds = fit_conditional_gcomp(
        x, z, y, SEED, iterations=GCOMP_ITERS, draws=MC_DRAWS)
    paper_methods['conditional_engression_gcomp'] = gcomp_draws
    paper_times['conditional_engression_gcomp'] = gcomp_seconds

In [ ]:
paper_table = pd.concat([evaluate_joint_full(draws, method)
                        for method, draws in paper_methods.items()], ignore_index=True)
paper_table['fit_seconds'] = paper_table.method.map(paper_times)
event_table = pd.concat([evaluate_events(draws, method)
                        for method, draws in paper_methods.items()], ignore_index=True)
contrast_table = fixed_event_contrasts(event_table)
latent_rows = []
for arm in (0, 1):
    values = validate_draws(latent_draws[arm])
    mu, covariance = values.mean(axis=0), np.cov(values, rowvar=False)
    grid_mae, grid_max = cdf_grid_errors(values, np.zeros(2), np.eye(2))
    latent_rows.append(dict(arm=arm, mean1=mu[0], mean2=mu[1],
        var1=covariance[0, 0], var2=covariance[1, 1], cov12=covariance[0, 1],
        normal_cdf_grid_mae=grid_mae, normal_cdf_grid_max_error=grid_max))
latent_table = pd.DataFrame(latent_rows)

print('Moments: truth is (0,0) or (2,-1), unit variances and covariance 0.4')
display(paper_table.set_index(['method', 'arm'])[['estimated_mean1', 'estimated_mean2',
    'mean_mae', 'estimated_var1', 'estimated_var2', 'estimated_cov12',
    'cov12_error', 'covariance_fro_error']].round(4))
print('Marginal quantiles and joint probabilities on a prespecified grid')
display(paper_table.set_index(['method', 'arm'])[['marginal_quantile_mae',
    'joint_cdf_grid_mae', 'joint_cdf_grid_max_error', 'joint_event_mae', 'fit_seconds']].round(4))
print('Joint event probabilities; model-draw MCSE excludes model-estimation uncertainty')
display(event_table.set_index(['method', 'arm', 'event'])[['true_probability',
    'estimated_probability', 'probability_error', 'true_independent_probability',
    'true_dependence_gap', 'estimated_dependence_gap', 'model_draw_mcse']].round(4))
print('Treatment contrasts for the two events with fixed thresholds')
display(contrast_table.set_index(['method', 'event']).round(4))
print('Latent calibration under the empirical Z law: target N(0,I_2)')
display(latent_table.set_index('arm').round(4))
print('frengression_gcomp_audit checks the fitted f/h composition; it is not a competitor.')
print('The median-quadrant truth is 0.3155; exact independent margins give 0.25.')
print('A pilot describes one fitted dataset. Comparative claims require paired replications.')

In [ ]:
RESULT_DIR = Path.home() / 'frengression_results' / 'joint_outcomes_causl'
PILOT_OUTPUT = RESULT_DIR / f'pilot_seed{SEED}_fr{PAPER_ITERS}_mc{MC_DRAWS}'
SAVE_PILOT = True

def installed_version(name):
    try:
        return package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        return None

git_info = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'],
                          capture_output=True, text=True)
run_metadata = dict(n=N, seed=SEED, instrument_strength=STRENGTH_INSTR,
    paper_fit=PAPER_FIT, frengression_updates=PAPER_ITERS, gcomp_updates=GCOMP_ITERS,
    gcomp_pilot_enabled=RUN_GCOMP_PILOT, mc_draws=MC_DRAWS, noise_dim=2,
    layers=3, width=100, learning_rate=1e-4,
    true_mean={str(k): v.tolist() for k, v in TRUE_MEAN.items()},
    true_covariance=TRUE_COV.tolist(), fixed_events=FIXED_EVENTS,
    quantile_levels=QUANTILE_LEVELS.tolist(), cdf_grid_offsets=CDF_GRID_OFFSETS.tolist(),
    repository_commit=git_info.stdout.strip() if git_info.returncode == 0 else None,
    python=platform.python_version(), platform=platform.platform(),
    versions={name: installed_version(name) for name in
              ['torch', 'engression', 'numpy', 'pandas', 'scipy']},
    uncertainty_note='model_draw_mcse is Monte Carlo variability conditional on the fitted model',
    timing_note='outcome-module fitting only; excludes R generation and complete simulator fitting')
if SAVE_PILOT:
    PILOT_OUTPUT.mkdir(parents=True, exist_ok=True)
    for name, table in [('moments_and_joint_scores', paper_table), ('joint_events', event_table),
                        ('fixed_event_contrasts', contrast_table), ('latent_calibration', latent_table),
                        ('sampling_audit', diagnostic_table)]:
        table.to_csv(PILOT_OUTPUT / f'{name}.csv', index=False)
    (PILOT_OUTPUT / 'metadata.json').write_text(json.dumps(run_metadata, indent=2))
    print('Saved pilot report:', PILOT_OUTPUT)

### What these outputs can support

Use the moment, quantile, grid and joint-event tables to report how accurately
the fitted model recovers this specified **bivariate interventional law**.
The shuffled diagnostic holds both fitted marginal distributions fixed; a
larger joint error there demonstrates the information lost by discarding
outcome dependence. Its same-marginal comparison is an ablation, not a fitted
method or a Frugal Flows result. The f/h audit checks compatibility under the
current covariate law. Neither the audit nor direct causal-margin sampling
evaluates a separately fitted covariate generator or a complete simulator.

The fitted conditional-engression baseline asks whether explicit marginal
modeling helps relative to fitting the joint conditional outcome law and
averaging over covariates. Report its actual repeated errors, including any
settings in which it performs better. The fixed-event contrast is a causal
contrast; the above-median event is an arm-specific dependence diagnostic.

This setting has the **same** true covariance in both arms. It demonstrates
dependence recovery under each intervention, not a treatment effect on
dependence. It also concerns two outcomes under the same intervention, not
the unidentified cross-world coupling of outcomes under different treatments.

In [ ]:
# Compare joint draws and shuffled margins against the exact 95% Gaussian ellipse.
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
grid = np.linspace(0, 2 * np.pi, 250)
eigvals, eigvecs = np.linalg.eigh(TRUE_COV)
unit_circle = np.stack((np.cos(grid), np.sin(grid)))
ell = (eigvecs @ np.diag(np.sqrt(eigvals)) @ unit_circle) * np.sqrt(-2 * np.log(0.05))
for row, method in enumerate(['frengression_joint', 'shuffled_diagnostic']):
    for arm in (0, 1):
        ax = axes[row, arm]
        d = paper_methods[method][arm]
        subset = np.random.default_rng(SEED + arm).choice(len(d), size=min(700, len(d)), replace=False)
        ax.scatter(d[subset, 0], d[subset, 1], alpha=0.18, s=8, label=method)
        contour = ell + TRUE_MEAN[arm][:, None]
        ax.plot(contour[0], contour[1], color='black', lw=2, label='Exact 95% ellipse')
        ax.set(title=f'{method}: do(A={arm})', xlabel='Y1', ylabel='Y2')
        ax.set_aspect('equal', adjustable='datalim')
        ax.legend(loc='upper left', fontsize=8)
if SAVE_PILOT:
    fig.savefig(PILOT_OUTPUT / 'joint_dependence.png', dpi=180, bbox_inches='tight')
plt.show()

## Paired replications (off by default)

Set `RUN_REPEATS=True` after inspecting the DGP, sampling and calibration
checks. `N_REPEATS=5` is a small protocol check; increase it to 30 for the final
comparison. The selected `PAPER_ITERS` is passed explicitly to every fit.
`RUN_GCOMP_REPEATS=True` fits the conditional-engression baseline on each same
observational dataset. Fix both fitting budgets before running this block.

Tables are saved after each seed outside the repository. Failures are retained
in a separate file and counted. Summaries report the number of completed
seeds, mean errors, across-seed SD and the Monte Carlo SE of their mean.
Paired differences are comparator error minus Frengression error, so positive
values favor Frengression. Only complete pairs enter those differences; check
the failure counts alongside them. These summaries do not provide confidence
intervals for a causal quantity estimated on one real dataset.

In [ ]:
RUN_REPEATS = False
N_REPEATS = 5
RUN_GCOMP_REPEATS = True
SUMMARY_METRICS = ['mean_mae', 'cov12_error', 'covariance_fro_error',
                   'marginal_quantile_mae', 'joint_cdf_grid_mae', 'joint_event_mae']

def summarize_errors(table, group_keys, metrics):
    rows = []
    for key, group in table.groupby(group_keys, sort=False):
        values = key if isinstance(key, tuple) else (key,)
        for metric in metrics:
            scores = group[metric].dropna().to_numpy(dtype=float)
            if not len(scores):
                continue
            row = dict(zip(group_keys, values))
            sd = float(np.std(scores, ddof=1)) if len(scores) > 1 else np.nan
            row.update(metric=metric, n=len(scores), mean=float(scores.mean()),
                       sd=sd, mcse_of_mean=sd / np.sqrt(len(scores)))
            rows.append(row)
    return pd.DataFrame(rows)

def paired_error_summary(table, comparator, metrics=SUMMARY_METRICS):
    fr = table.loc[table.method == 'frengression_joint'].set_index(['seed', 'arm'])
    other = table.loc[table.method == comparator].set_index(['seed', 'arm'])
    if not fr.index.is_unique or not other.index.is_unique:
        raise ValueError('Repeated method/seed/arm scores')
    common = fr.index.intersection(other.index)
    if not len(common):
        return pd.DataFrame()
    differences = (other.loc[common, metrics] - fr.loc[common, metrics]).reset_index()
    summary = summarize_errors(differences, ['arm'], metrics)
    summary.insert(0, 'comparator', comparator)
    return summary.rename(columns={'n': 'n_pairs', 'mean': 'mean_error_difference'})

if RUN_REPEATS:
    if N_REPEATS < 2:
        raise ValueError('At least two independent datasets are needed for across-seed variability')
    repeats_dir = RESULT_DIR / f'repeats_n{N}_fr{PAPER_ITERS}_gcomp{GCOMP_ITERS}_k{N_REPEATS}'
    repeats_dir.mkdir(parents=True, exist_ok=True)
    repeat_metadata = {**run_metadata, 'n_repeats': N_REPEATS,
                       'gcomp_repeats_enabled': RUN_GCOMP_REPEATS,
                       'seeds': list(range(SEED, SEED + N_REPEATS))}
    (repeats_dir / 'metadata.json').write_text(json.dumps(repeat_metadata, indent=2))
    all_scores, all_events, failures = [], [], []
    for seed in range(SEED, SEED + N_REPEATS):
        try:
            xx, zz, yy, _ = generate_causl_joint(N, seed, STRENGTH_INSTR)
        except Exception as error:
            failures.append(dict(seed=seed, method='data_generation', error=str(error)))
            pd.DataFrame(failures).to_csv(repeats_dir / 'failures.csv', index=False)
            continue
        methods, times = {}, {}
        try:
            _, draws, seconds = fit_joint(xx, zz, yy, seed, iterations=PAPER_ITERS, draws=MC_DRAWS)
            methods['frengression_joint'], times['frengression_joint'] = draws, seconds
            methods['shuffled_diagnostic'] = shuffle_second_coordinate(draws, seed + 99)
            times['shuffled_diagnostic'] = np.nan
        except Exception as error:
            failures.append(dict(seed=seed, method='frengression_joint', error=str(error)))
        if RUN_GCOMP_REPEATS:
            try:
                _, draws, seconds = fit_conditional_gcomp(xx, zz, yy, seed,
                                                          iterations=GCOMP_ITERS, draws=MC_DRAWS)
                methods['conditional_engression_gcomp'], times['conditional_engression_gcomp'] = draws, seconds
            except Exception as error:
                failures.append(dict(seed=seed, method='conditional_engression_gcomp', error=str(error)))
        for method, draws in methods.items():
            try:
                scores, events = evaluate_joint_full(draws, method), evaluate_events(draws, method)
                scores.insert(0, 'seed', seed)
                scores['fit_seconds'] = times[method]
                events.insert(0, 'seed', seed)
                all_scores.append(scores)
                all_events.append(events)
            except Exception as error:
                failures.append(dict(seed=seed, method=method, error=str(error)))
        # Keep every completed seed and every failure, even if a later fit is interrupted.
        if all_scores:
            pd.concat(all_scores, ignore_index=True).to_csv(repeats_dir / 'scores.csv', index=False)
            pd.concat(all_events, ignore_index=True).to_csv(repeats_dir / 'events.csv', index=False)
        pd.DataFrame(failures, columns=['seed', 'method', 'error']).to_csv(repeats_dir / 'failures.csv', index=False)
    if not all_scores:
        raise RuntimeError(f'No completed fits; inspect {repeats_dir / "failures.csv"}')
    repeated_results = pd.concat(all_scores, ignore_index=True)
    repeated_events = pd.concat(all_events, ignore_index=True)
    repeated_contrasts = fixed_event_contrasts(repeated_events)
    score_summary = summarize_errors(repeated_results, ['method', 'arm'], SUMMARY_METRICS)
    event_summary = summarize_errors(repeated_events, ['method', 'arm', 'event'], ['probability_error'])
    contrast_summary = summarize_errors(repeated_contrasts, ['method', 'event'],
                                        ['contrast_error', 'signed_contrast_error'])
    paired_tables = [paired_error_summary(repeated_results, comparator)
                     for comparator in ['shuffled_diagnostic', 'conditional_engression_gcomp']]
    paired_tables = [table for table in paired_tables if not table.empty]
    paired_summary = pd.concat(paired_tables, ignore_index=True) if paired_tables else pd.DataFrame()
    for name, table in [('score_summary', score_summary), ('event_summary', event_summary),
                        ('contrast_summary', contrast_summary), ('paired_error_summary', paired_summary),
                        ('fixed_event_contrasts', repeated_contrasts)]:
        table.to_csv(repeats_dir / f'{name}.csv', index=False)
    print(f'Planned datasets: {N_REPEATS}; failures: {len(failures)}')
    display(pd.DataFrame(failures, columns=['seed', 'method', 'error']))
    display(score_summary.round(4))
    display(event_summary.round(4))
    display(contrast_summary.round(4))
    print('Paired differences: comparator error minus Frengression error; positive favors Frengression.')
    display(paired_summary.round(4))
    print('Saved repeated report:', repeats_dir)